In [ ]:
# Chong phan manh bo nho GPU. PHAI dat TRUOC lan cap phat CUDA dau tien.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q "transformers>=4.40" "sentence-transformers>=3.0.0"
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU")
print("DONE")


In [ ]:
# FINE-TUNE CROSS-ENCODER — cong thuc arXiv:2507.14619 (Semi-Hard Negative Mining, UIT).
# Negative lay NGAU NHIEN trong ro, KHONG phai top-n kho nhat: bang 2 cua ho cho
# hard 0.6751 vs semi-hard 0.7911 MRR@10. n=10, BCEWithLogitsLoss, lr 2e-5.
import os, glob, gzip, json, time, random, math
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

RERANK_MODEL = "AITeamVN/Vietnamese_Reranker"
OUT_DIR      = "/kaggle/working/reranker_ft"

MAX_LEN       = 512  # khop voi baseline
BATCH_PER_GPU = 4    # ha tu 8 sau khi OOM; cell 5 cho biet co tang lai duoc khong
ACCUM         = 4    # 2x T4 -> batch that = 8, hieu dung = 32
GROUP_BY_LENGTH = True   # xem cell 3
FREEZE_EMB    = True     # dong bang bang embedding, xem cell 4
GRAD_CKPT     = False    # bat neu VAN OOM: giam manh bo nho activation, cham them ~30%
LR        = 2e-5   # theo bai bao
EPOCHS    = 2      # bai bao ghi 2, nhung ca 2 lan train deu overfit o epoch 2 -> 1 la du
WARMUP    = 0.05
SEED      = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU  = torch.cuda.device_count()
BATCH  = BATCH_PER_GPU * max(N_GPU, 1)   # DataParallel chia batch nay cho tung GPU

# Kaggle TU GIAI NEN file .gz khi upload -> phai chap nhan ca hai duoi
def find_pairs(name = "train_pairs_semi_n10"):
    for pat in (f"/kaggle/input/**/{name}.jsonl.gz", f"/kaggle/input/**/{name}.jsonl"):
        hits = sorted(glob.glob(pat, recursive = True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Khong thay {name}.jsonl(.gz) trong /kaggle/input")

# Uu tien file pool; khong co thi quay ve file cu. Nhin dong in ra de biet dang train file nao.
try:
    PAIRS = find_pairs("train_pairs_pool_n10")
    print(">>> DANG DUNG NEGATIVE DUNG PHAN PHOI (pool)")
except FileNotFoundError:
    PAIRS = find_pairs("train_pairs_semi_n10")
    print(">>> CANH BAO: khong thay file pool, dang dung file CU (BM25-only)")
print("du lieu:", PAIRS, f"({os.path.getsize(PAIRS)/1e6:.1f} MB)")
print("device :", DEVICE, f"| {N_GPU} GPU | batch {BATCH} ({BATCH_PER_GPU}/GPU), hieu dung {BATCH*ACCUM}")

# Chot chan: chay hong mot lan (OOM) thi model + AdamW state cua lan truoc VAN nam
# trong RAM GPU. Khong restart thi lan nao cung OOM, ma traceback lai chi vao cho khac.
if DEVICE == "cuda":
    import gc
    gc.collect(); torch.cuda.empty_cache()
    _used = torch.cuda.memory_allocated(0) / 2**30
    _free, _cap = (x / 2**30 for x in torch.cuda.mem_get_info(0))
    print(f"GPU 0: {_free:.2f}/{_cap:.2f} GB trong | tien trinh nay giu {_used:.2f} GB")
    assert _used < 1.0 and _free > _cap * 0.75, (
        f"GPU chua sach ({_used:.2f} GB bi giu) -> Run > Restart Session roi chay lai.")


In [ ]:
def load_rows(path):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rt", encoding = "utf-8") as f:
        return [json.loads(l) for l in f]

t0 = time.time()
rows = load_rows(PAIRS)
train_rows = [r for r in rows if r["split"] == "train"]
dev_rows   = [r for r in rows if r["split"] == "dev"]

def summarize(name, rs):
    pos = sum(r["label"] == 1 for r in rs)
    q   = len({r["qid"] for r in rs})
    print(f"  {name:<6} {len(rs):>7,} dong | {q:>6,} cau | positive {pos:>6,} | negative {len(rs)-pos:>6,}")

print(f"nap {len(rows):,} dong trong {time.time()-t0:.0f}s")
summarize("train", train_rows); summarize("dev", dev_rows)

# Chot chan an toan: dev PHAI roi han train theo qid, va khong cau nao duoc trung warmup.
assert not ({r["qid"] for r in train_rows} & {r["qid"] for r in dev_rows}), "dev bi lan sang train"

WARMUP_PATH = sorted(glob.glob("/kaggle/input/**/warmup_IR.json", recursive = True))
if WARMUP_PATH:
    wq = {str(k) for k in json.load(open(WARMUP_PATH[0], encoding = "utf-8"))}
    leak = {r["qid"] for r in rows} & wq
    print(f"\nro ri warmup: {len(leak)} cau  ->  {'SACH' if not leak else 'CO RO RI, DUNG LAI'}")
    assert not leak, "Du lieu train chua cau cua warmup -> diem warmup se la lac quan gia"
else:
    print("\n(khong tim thay warmup_IR.json de kiem tra ro ri)")


In [ ]:
tok = AutoTokenizer.from_pretrained(RERANK_MODEL)

class PairDS(Dataset):
    def __init__(self, rs): self.rs = rs
    def __len__(self):      return len(self.rs)
    def __getitem__(self, i):
        r = self.rs[i]
        return r["question"], r["text"], float(r["label"])

def collate(batch):
    q, t, y = zip(*batch)
    enc = tok(list(q), list(t), truncation = True, max_length = MAX_LEN,
              padding = True, return_tensors = "pt")
    enc["labels"] = torch.tensor(y, dtype = torch.float32)
    return enc

class LengthGroupedBatches(torch.utils.data.Sampler):
    """Gom cap cung do dai vao mot batch: pad den ~290 token thay vi ~500, do ~40% phep tinh.

    Do dai uoc bang so KY TU nen khong phai chay tokenizer truoc. Van xao tron: xao
    trong tung khoi lon truoc khi sap, va xao ca THU TU cac batch.
    """
    def __init__(self, rows, batch_size, seed = 0, mega = 50):
        self.lens = [len(r["question"]) + len(r["text"]) for r in rows]
        self.bs, self.seed, self.mega, self.epoch = batch_size, seed, mega * batch_size, 0

    def set_epoch(self, e): self.epoch = e

    def __iter__(self):
        g = random.Random(self.seed + self.epoch)
        idx = list(range(len(self.lens)))
        g.shuffle(idx)
        batches = []
        for i in range(0, len(idx), self.mega):
            block = sorted(idx[i:i+self.mega], key = lambda j: self.lens[j])
            batches += [block[j:j+self.bs] for j in range(0, len(block), self.bs)]
        batches = [b for b in batches if len(b) == self.bs]
        g.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return len(self.lens) // self.bs


if GROUP_BY_LENGTH:
    train_sampler = LengthGroupedBatches(train_rows, BATCH, seed = SEED)
    train_dl = DataLoader(PairDS(train_rows), batch_sampler = train_sampler,
                          collate_fn = collate, num_workers = 2)
else:
    train_sampler = None
    train_dl = DataLoader(PairDS(train_rows), batch_size = BATCH, shuffle = True,
                          collate_fn = collate, num_workers = 2, drop_last = True)

# dev sap theo do dai luon: khong anh huong ket qua, chi nhanh hon
dev_sorted = sorted(range(len(dev_rows)), key = lambda i: len(dev_rows[i]["question"]) + len(dev_rows[i]["text"]))
dev_rows_s = [dev_rows[i] for i in dev_sorted]
dev_dl = DataLoader(PairDS(dev_rows_s), batch_size = BATCH * 2, shuffle = False,
                    collate_fn = collate, num_workers = 2)
print(f"{len(train_dl):,} step/epoch | gom theo do dai: {GROUP_BY_LENGTH}")


In [ ]:
@torch.no_grad()
def score_all(model, dl):
    """Cham diem toan bo dataloader, tra ve mang logit."""
    model.eval()
    out = []
    for b in dl:
        y = b.pop("labels")
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast("cuda", dtype = torch.float16, enabled = DEVICE == "cuda"):
            out.append(model(**b).logits.squeeze(-1).float().cpu())
    model.train()
    return torch.cat(out).numpy()


def dev_acc(scores, rs):
    """Ti le cau dev ma ung vien diem CAO NHAT la positive.

    PROXY re tren 11 ung vien/cau. Con so that phai do bang full pipeline tren warmup.
    """
    by_q = {}
    for s, r in zip(scores, rs):
        by_q.setdefault(r["qid"], []).append((s, r["label"]))
    hit = sum(1 for v in by_q.values() if max(v, key = lambda x: x[0])[1] == 1)
    return hit / len(by_q), len(by_q)


base_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL, num_labels = 1).to(DEVICE)

# Bang embedding XLM-R = 256M tham so = 45% ca model; gradient fp32 cua rieng no
# = 977 MiB, dung con so trong loi "Tried to allocate 978.00 MiB". Dong bang no ha
# bo nho thuong tru 8,96 -> 5,88 GB, va khong mat gi vi tokenizer khong doi.
if FREEZE_EMB:
    _emb = getattr(base_model.base_model, "embeddings", None) or base_model.get_input_embeddings()
    for p in _emb.parameters():
        p.requires_grad = False

if GRAD_CKPT:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs = {"use_reentrant": False})

TRAIN_PARAMS = [p for p in base_model.parameters() if p.requires_grad]
_n_all = sum(p.numel() for p in base_model.parameters())
_n_tr  = sum(p.numel() for p in TRAIN_PARAMS)
print(f"tham so: {_n_all/1e9:.3f} ty | huan luyen {_n_tr/1e9:.3f} ty ({_n_tr/_n_all*100:.0f}%)"
      f" | dong bang embedding: {FREEZE_EMB} | grad checkpoint: {GRAD_CKPT}")

# DataParallel: PyTorch thuan, khong dinh den API multi-process cua sentence-transformers.
# Loi ~1,5x chu khong phai 2x: moi buoc phai broadcast 2,3 GB trong so sang GPU thu hai.
if N_GPU > 1:
    model = torch.nn.DataParallel(base_model)
    print(f"DataParallel tren {N_GPU} GPU")
else:
    model = base_model
    print("chi 1 GPU")

# PHAI dung dev_rows_s chu khong phai dev_rows: dev_dl duyet ban DA SAP theo do dai,
# dung nham thi diem va nhan lech nhau -> con so vo nghia ma khong bao loi nao.
s0 = score_all(model, dev_dl)
a0, nq = dev_acc(s0, dev_rows_s)
print(f"\ndev accuracy@1 TRUOC khi train: {a0:.4f}  ({nq} cau)")


In [ ]:
# Thu forward + backward tren batch NANG NHAT co the: ton 20 giay, doi lai khong
# phai chay 1,5 gio roi moi biet OOM.
import gc
base_model.zero_grad(set_to_none = True)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

_ds  = PairDS(train_rows)
_wst = sorted(range(len(train_rows)),
              key = lambda i: -(len(train_rows[i]["question"]) + len(train_rows[i]["text"])))[:BATCH]
_b   = collate([_ds[i] for i in _wst])
_y   = _b.pop("labels").to(DEVICE)
_b   = {k: v.to(DEVICE) for k, v in _b.items()}
print(f"batch nang nhat: {tuple(_b['input_ids'].shape)}  (batch x token)")

model.train()
_sc = torch.amp.GradScaler("cuda", enabled = DEVICE == "cuda")
for _ in range(2):                      # 2 vong de bat ca dinh cua backward tich luy
    with torch.autocast("cuda", dtype = torch.float16, enabled = DEVICE == "cuda"):
        _l = torch.nn.BCEWithLogitsLoss()(model(**_b).logits.squeeze(-1).float(), _y) / ACCUM
    _sc.scale(_l).backward()

_peak = torch.cuda.max_memory_allocated(0) / 2**30
_adam = _n_tr * 8 / 2**30               # exp_avg + exp_avg_sq, fp32
_tot  = _peak + _adam
_cap  = torch.cuda.get_device_properties(0).total_memory / 2**30

base_model.zero_grad(set_to_none = True)
del _b, _y, _l, _sc, _ds
gc.collect(); torch.cuda.empty_cache()

print(f"uoc tinh GPU 0: {_tot:.2f}/{_cap:.2f} GB  (dinh {_peak:.2f} + AdamW {_adam:.2f})")
print("  => DU CHO" if _tot < _cap * 0.85 else
      f"  => CHAT QUA: dat GRAD_CKPT=True hoac ha BATCH_PER_GPU xuong {max(BATCH_PER_GPU//2,1)}")


In [ ]:
opt   = torch.optim.AdamW(TRAIN_PARAMS, lr = LR, weight_decay = 0.01)   # chi param con requires_grad
steps = len(train_dl) // ACCUM * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(steps * WARMUP), steps)
scaler = torch.amp.GradScaler("cuda", enabled = DEVICE == "cuda")
lossf = torch.nn.BCEWithLogitsLoss()

print(f"tong {steps:,} step toi uu | {len(train_dl):,} batch/epoch, uoc tinh ~1,5-2 gio tren 2x T4\n")
t0, run, best = time.time(), 0.0, -1.0
model.train()
for ep in range(EPOCHS):
    if train_sampler is not None:
        train_sampler.set_epoch(ep)   # doi cach gom batch moi epoch
    for i, b in enumerate(train_dl):
        y = b.pop("labels").to(DEVICE)
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast("cuda", dtype = torch.float16, enabled = DEVICE == "cuda"):
            logit = model(**b).logits.squeeze(-1)
            loss = lossf(logit.float(), y) / ACCUM
        scaler.scale(loss).backward()
        run += loss.item() * ACCUM
        if (i + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(TRAIN_PARAMS, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none = True); sched.step()
        if (i + 1) % 1000 == 0:
            el = time.time() - t0
            done = ep * len(train_dl) + i + 1
            tot = EPOCHS * len(train_dl)
            print(f"  ep{ep+1} {i+1:>6,}/{len(train_dl):,} | loss {run/1000:.4f} "
                  f"| {el/60:.0f}p da qua, con ~{(el/done*(tot-done))/60:.0f}p", flush = True)
            run = 0.0

    acc, _ = dev_acc(score_all(model, dev_dl), dev_rows_s)
    print(f"\n=== het epoch {ep+1}: dev accuracy@1 = {acc:.4f}  (truoc khi train {a0:.4f}) ===\n")
    if acc > best:
        best = acc
        base_model.save_pretrained(OUT_DIR, safe_serialization = True)  # .module, khong phai wrapper
        tok.save_pretrained(OUT_DIR)
        print(f"    -> tot nhat tu truoc toi nay, da luu vao {OUT_DIR}")

print(f"\nXONG sau {(time.time()-t0)/60:.0f} phut | dev accuracy@1 tot nhat: {best:.4f}")


In [ ]:
# --- Kiem tra model da luu ---
import os
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:<32} {os.path.getsize(os.path.join(OUT_DIR,f))/1e6:>8.1f} MB")

print("\n-> Quick Save, roi attach output nay vao baseline (go output finetune CU ra).")
print("   Diem that do bang eval_official tren warmup, KHONG phai dev accuracy@1.")